In [1]:
from pathlib import Path
import re
import pandas as pd

PDF_PATH = "jofi13377-sup-0001-internetappendix.pdf"   # <- your input PDF
PAGES = "8-25"  # Table IA.I pages in the PDF (document page numbers)

OUT_CSV = "wsj_topics_ia_c_table_IAI.csv"
OUT_CSV_FIRST5 = "wsj_topics_ia_c_first5.csv"


def _normalize_spaces(s: str) -> str:
    s = s.replace("\u00ad", "")  # soft hyphen
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()


def _first_five_terms(key_terms: str) -> str:
    # Split on commas; keep first 5 non-empty items
    parts = [p.strip() for p in key_terms.split(",") if p.strip()]
    return ", ".join(parts[:5])


def extract_with_camelot(pdf_path: str, pages: str) -> pd.DataFrame:
    """
    Extract tables using Camelot (preferred).
    Produces columns: metatopic_label, topic_label, key_terms.
    """
    import camelot  # pip install camelot-py

    # "stream" works well for tables without explicit ruling lines
    tables = camelot.read_pdf(pdf_path, pages=pages, flavor="stream")

    if len(tables) == 0:
        raise RuntimeError("Camelot found no tables on the specified pages.")

    # Concatenate all extracted table chunks
    raw = pd.concat([t.df for t in tables], ignore_index=True)

    # Drop obvious header rows
    # Camelot often returns headers as a row containing these strings
    header_mask = raw.apply(
        lambda r: any("Metatopic label" in str(x) or "Topic label" in str(x) or "Key terms" in str(x) for x in r),
        axis=1
    )
    raw = raw.loc[~header_mask].copy()

    # Camelot sometimes returns 3 columns, sometimes more; we want first 3 meaningful columns.
    # Heuristic: keep the left-most 3 columns.
    raw = raw.iloc[:, :3]
    raw.columns = ["metatopic_label", "topic_label", "key_terms"]

    # Clean text
    for c in raw.columns:
        raw[c] = raw[c].astype(str).map(_normalize_spaces)

    # Remove empty rows
    raw = raw[~((raw["metatopic_label"] == "") & (raw["topic_label"] == "") & (raw["key_terms"] == ""))].copy()

    # Merge wrapped rows:
    # Often, a row continues with empty topic_label but extra key_terms text.
    rows = []
    current = None

    for _, r in raw.iterrows():
        m, t, k = r["metatopic_label"], r["topic_label"], r["key_terms"]

        # Skip page continuation markers
        if "Continued on next page" in m or "Continued on next page" in t or "Continued on next page" in k:
            continue
        if m.startswith("Table IA.I"):
            continue

        if t and t.lower() not in {"nan", "none"}:
            # Start a new row
            if current is not None:
                rows.append(current)
            current = {"metatopic_label": m, "topic_label": t, "key_terms": k}
        else:
            # Continuation of previous row (append key terms)
            if current is None:
                # If Camelot messed up and we got continuation before any start, skip
                continue
            if k:
                if current["key_terms"]:
                    current["key_terms"] = (current["key_terms"] + " " + k).strip()
                else:
                    current["key_terms"] = k

            # Sometimes metatopic shows up on continuation lines; keep first non-empty
            if (not current["metatopic_label"]) and m:
                current["metatopic_label"] = m

    if current is not None:
        rows.append(current)

    df = pd.DataFrame(rows)

    # Final cleaning: remove any lingering "nan"
    for c in df.columns:
        df[c] = df[c].replace({"nan": "", "None": ""}).map(_normalize_spaces)

    # Filter rows that look like real topics (need topic_label + key_terms with commas)
    df = df[(df["topic_label"] != "") & (df["key_terms"].str.contains(",", regex=False))].copy()

    return df.reset_index(drop=True)


def extract_with_pdfplumber_fallback(pdf_path: str, pages: str) -> pd.DataFrame:
    """
    Fallback: extract text and parse rows heuristically.
    This is less perfect than Camelot but works if table extraction fails.
    """
    import pdfplumber  # pip install pdfplumber

    # Convert "8-24" -> list of 0-based page indices (pdfplumber is 0-based)
    start, end = [int(x) for x in pages.split("-")]
    page_idxs = list(range(start - 1, end))  # inclusive end

    chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for i in page_idxs:
            txt = pdf.pages[i].extract_text() or ""
            chunks.append(txt)

    text = "\n".join(chunks)
    text = text.replace("\u00ad", "")  # soft hyphen
    text = re.sub(r"\n+", "\n", text)

    # Remove obvious headers/footers
    drop_patterns = [
        r"^Table IA\.I.*$",
        r"^Table IA\.I.*Continued.*$",
        r"^Metatopic label Topic label Key terms$",
        r"^Continued on next page$",
    ]
    for pat in drop_patterns:
        text = re.sub(pat, "", text, flags=re.MULTILINE)

    # Heuristic parse:
    # Many lines look like:
    #   <Metatopic> <Topic label> <term1>, <term2>, ...
    # We look for lines that contain a comma-separated term list,
    # then split the left side into metatopic/topic by taking first token group as metatopic,
    # second as topic label, rest as key terms.
    rows = []
    for line in text.splitlines():
        line = _normalize_spaces(line)
        if not line:
            continue
        if "," not in line:
            continue

        # Try to split into 3 parts by using the first occurrence of a "term-like" pattern:
        # We assume key terms start at the first comma-containing segment.
        # So find the position of the first comma, then backtrack to find start of that term list.
        comma_pos = line.find(",")
        left = line[:comma_pos].strip()
        right = line[comma_pos - 0 :].strip()  # include from comma onward

        # left is "<Metatopic> <Topic label> <firstterm_without_comma>"
        # right starts with ", ..." so we need first term too.
        # We reconstruct by taking the last "word-ish" chunk in left as first term candidate.
        # Better: split left into tokens and treat last token group as first key term piece.
        toks = left.split(" ")
        if len(toks) < 3:
            continue

        # Assume metatopic = first token group, topic label = next token group(s) until we hit term-like start.
        # This is ambiguous; fallback to: metatopic = first token, topic_label = second token,
        # first_term_piece = remaining tokens joined.
        metatopic = toks[0]
        topic_label = toks[1]
        first_term_piece = " ".join(toks[2:]).strip()

        key_terms = f"{first_term_piece}{right}"
        key_terms = _normalize_spaces(key_terms)

        rows.append(
            {
                "metatopic_label": metatopic,
                "topic_label": topic_label,
                "key_terms": key_terms,
            }
        )

    df = pd.DataFrame(rows).drop_duplicates()

    # Filter: keep only plausible rows (key_terms should have multiple commas)
    df = df[df["key_terms"].str.count(",") >= 3].copy()
    df.reset_index(drop=True, inplace=True)
    return df


def main():
    pdf_path = str(PDF_PATH)
    if not Path(pdf_path).exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    # 1) Try Camelot (best)
    try:
        df = extract_with_camelot(pdf_path, PAGES)
        method = "camelot"
    except Exception as e:
        print(f"[WARN] Camelot extraction failed: {e}")
        print("[WARN] Falling back to pdfplumber heuristic parsing (may need manual cleanup).")
        df = extract_with_pdfplumber_fallback(pdf_path, PAGES)
        method = "pdfplumber_fallback"

    print(f"[OK] Extracted {len(df)} rows using: {method}")

    # Add first-5 terms column (what you need for your prompt)
    df["first_5_terms"] = df["key_terms"].map(_first_five_terms)

    # Save outputs
    df.to_csv(OUT_CSV, index=False)
    df[["topic_label", "first_5_terms"]].to_csv(OUT_CSV_FIRST5, index=False)

    print(f"[OK] Wrote: {OUT_CSV}")
    print(f"[OK] Wrote: {OUT_CSV_FIRST5}")

    # Quick sanity check: show a few rows
    print(df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()

c:\Users\jonat\anaconda3\envs\jupyter311\Lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


[OK] Extracted 170 rows using: camelot
[OK] Wrote: wsj_topics_ia_c_table_IAI.csv
[OK] Wrote: wsj_topics_ia_c_first5.csv
metatopic_label                           topic_label                                                                                                                                                                                                                                                                                                              key_terms                                                             first_5_terms
                                 Mid-level executives              vice, vice president, executive vice, senior vice, named senior, named vice, named executive, president finance, president market, group vice, company vice, vice chairman, named president, group executive, newly create, business development, assistant vice, general manager, named, group president           vice, vice president, executive vice, senior vice, named senior
  

In [ ]:
# load csv with keywords for topics
keywords = pd.read_csv("C:\\Users\\jonat\\Lasso_paper\\narratives_construction\\scripts\\wsj_topics_ia_c_table_IAI.csv")

NameError: name 'keywords_df' is not defined

In [16]:
# in row 3, change topic_label to "Natural disasters"
keywords_df.iloc[2, keywords_df.columns.get_loc("topic_label")] = "Natural disasters"
